In [1]:
import pandas as pd
import numpy as np
import nltk
nltk.download('punkt')
from nltk.corpus import stopwords


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/dasariyaswanthsribalachandra/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [2]:
df = pd.read_csv('IMDB Dataset.csv')

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df.describe(include='all')

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [8]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [9]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [13]:
df['review']=df['review'].astype(str)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [15]:
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)          # remove HTML
    text = re.sub(r'[^a-zA-Z\s]', '', text)   # remove special chars
    text = text.lower()                       # lowercase
    return text

df['review'] = df['review'].apply(clean_text)

In [16]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [17]:
df['review'] = df['review'].apply(lambda x: x.split())

In [18]:
df.head()

,review,sentiment
0,"[one, of, the, other, reviewers, has, mentione...",positive
1,"[a, wonderful, little, production, the, filmin...",positive
2,"[i, thought, this, was, a, wonderful, way, to,...",positive
3,"[basically, theres, a, family, where, a, littl...",negative
4,"[petter, matteis, love, in, the, time, of, mon...",positive


In [19]:
df['review'].isnull().sum()

np.int64(0)

In [20]:
df['review_length'] = df['review'].apply(len)

In [21]:
df.head()

,review,sentiment,review_length
0,"[one, of, the, other, reviewers, has, mentione...",positive,300
1,"[a, wonderful, little, production, the, filmin...",positive,156
2,"[i, thought, this, was, a, wonderful, way, to,...",positive,161
3,"[basically, theres, a, family, where, a, littl...",negative,128
4,"[petter, matteis, love, in, the, time, of, mon...",positive,222


In [24]:
df['review'] = df['review'].apply(lambda x: " ".join(x))

In [25]:
df.head()

,review,sentiment,review_length
0,one of the other reviewers has mentioned that ...,positive,300
1,a wonderful little production the filming tech...,positive,156
2,i thought this was a wonderful way to spend ti...,positive,161
3,basically theres a family where a little boy j...,negative,128
4,petter matteis love in the time of money is a ...,positive,222


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['review'])

In [27]:
df.head()

,review,sentiment,review_length
0,one of the other reviewers has mentioned that ...,positive,300
1,a wonderful little production the filming tech...,positive,156
2,i thought this was a wonderful way to spend ti...,positive,161
3,basically theres a family where a little boy j...,negative,128
4,petter matteis love in the time of money is a ...,positive,222


In [30]:
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [31]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['sentiment'])

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [34]:
y_pred = model.predict(X_test)

In [35]:
sample = ["this movie was fantastic and amazing"]

sample_vec = vectorizer.transform(sample)
prediction = model.predict(sample_vec)

print(le.inverse_transform(prediction))

['positive']


In [36]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[4391  570]
 [ 455 4584]]


In [37]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.89      0.90      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [40]:
# 1️⃣ Import libraries
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# 2️⃣ Load dataset
# Replace with your file path
df = pd.read_csv("IMDB Dataset.csv")   # columns: review, sentiment


# 3️⃣ Data Cleaning
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)          # remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)   # remove special characters
    text = text.lower()                       # lowercase
    return text

df['review'] = df['review'].apply(clean_text)


# 4️⃣ Feature Engineering (optional)
df['review_length'] = df['review'].apply(lambda x: len(x.split()))


# 5️⃣ Convert text → numerical (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['review'])


# 6️⃣ Encode labels (positive/negative → 1/0)
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])


# 7️⃣ Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 8️⃣ Train Model
model = LogisticRegression()
model.fit(X_train, y_train)


# 9️⃣ Prediction
y_pred = model.predict(X_test)


# 🔟 Evaluation

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Precision, Recall, F1-score
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# 1️⃣1️⃣ Test with custom input
sample = ["this movie was absolutely fantastic and amazing"]

sample_clean = [clean_text(sample[0])]
sample_vec = vectorizer.transform(sample_clean)

prediction = model.predict(sample_vec)

print("\nCustom Prediction:", le.inverse_transform(prediction)[0])

Accuracy: 0.8935

Confusion Matrix:
[[4370  591]
 [ 474 4565]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Custom Prediction: positive
